In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

#
np.random.seed(42)
torch.manual_seed(42)

# For Fictional dataset:
X, y = make_classification(n_samples=10000, n_features=40, n_informative=25, random_state=42)

**The Code to be debugged:**

In [ ]:
X = StandardScaler().fit_transform(X) #Bug 1: fit_transform before spliting data --> Data Leakage
X_tr, X_val = X[:8000], X[8000:]
y_tr, y_val = y[:8000], y[8000:]
np.random.shuffle(X_tr) #Bug 2: shuffle(X_tr), only shuffling X_tr lead to random noise

#Bug 3: No model.eval() and torch.no_grad()
#model.eval() helps the model use all neurons (when dropout removes the 50%)
#torch.no_grad helps in maintaining memry and time efficiently (by stopping PyTorch from keeping notes it won't use).


#Bug 4: placing Dropout(0.5) before BatchNorm1d(128)
model = nn.Sequential(nn.Linear(40, 128), nn.ReLU(), nn.Dropout(0.5), #Injects artificial variance before normalization
              nn.BatchNorm1d(128), nn.Linear(128, 2))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.9) #Bug 5: The learning rate shrinks by 10% so relentlessly that within a few rounds, your steps become microscopic.



for epoch in range(50):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()
    val_logits = model(torch.tensor(X_val)) #Bug 6: Numpy defaults to 64-bit precision whereas, pytorch defaults to 32-bit
    print(epoch, accuracy(val_logits, y_val))

**The corrected code**

In [ ]:
from sklearn.model_selection import train_test_split

# --- ACCURACY HELPER FUNCTION ---
def accuracy(logits, y_true):
    preds = torch.argmax(logits, dim=1)
    if not isinstance(y_true, torch.Tensor):
        y_true = torch.tensor(y_true)
    return (preds == y_true).float().mean().item()
# Fix 2: Split data before scaling values to keep X and y strictly synchronized (and stratified)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Fix 1: Fitted on training data only to prevent data leakage
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_val = sc.transform(X_val)
# Fix 6: Explicitly pass dtype=torch.float32 to avoid Float64 (Double) vs Float32 dtype mismatch
train_ds = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.long)
)
# PyTorch DataLoader handles random batch shuffling safely to prevent desynchronizing
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
# Fix 4: Corrected layer order (Linear -> BatchNorm -> ReLU -> Dropout)
model = nn.Sequential(
    nn.Linear(40, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),  # Placed after BatchNorm to avoid variance shift
    nn.Linear(128, 2)
)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# Fix 5: CosineAnnealingLR smoothly decays LR instead of aggressive step decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-5)
loss_fn = nn.CrossEntropyLoss()
# tarining & evaluation loop
for epoch in range(10):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(yb)
    # Step the learning rate scheduler every epoch
    scheduler.step()
    # Fix 3: Switch to model.eval() and torch.no_grad() to freeze Dropout and BatchNorm
    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val, dtype=torch.float32))
        val_acc = accuracy(val_logits, y_val)
    avg_train_loss = total_loss / len(train_ds)
    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc*100:.2f}%")